# PPLSTM vs sLSTM — two separate CUDA Graph paths (capture-fix experiment)

**Fresh 4-way rerun:** PPLSTM eager, PPLSTM CUDA Graph, official *vanilla* sLSTM eager, and official *vanilla* sLSTM with an experimental CUDA-capture-safe pointwise function. Each model gets its own graph object. Your PPLSTM implementation is **unchanged**.

### Why the earlier sLSTM graph failed

The [official xLSTM vanilla sLSTM pointwise function](https://github.com/NX-AI/xlstm/blob/main/xlstm/blocks/slstm/src/vanilla/slstm.py) executes `if torch.all(n == 0.0):`. That converts a GPU tensor into a Python condition, which synchronizes with the CPU and can invalidate CUDA Graph capture. In the graph-only sLSTM path, this notebook substitutes the algebraically equivalent `torch.where(torch.all(n == 0.0), iraw, torch.max(iraw, logfplusm))`, where both branches remain on the GPU. We verify original-vs-patched forward outputs and gradients on CPU before applying it to the sLSTM graph model. If validation or capture fails, it **reports failure**, rather than claiming a speedup.

The official sLSTM **optimized CUDA kernel** is still a different backend and is skipped on a Tesla T4 (compute capability 7.5). The graph patch does **not** turn vanilla sLSTM into the official optimized CUDA kernel.

**Run order:** Set Runtime → Change runtime type → GPU; then Runtime → Run all. `RUN_FULL_SWEEP=True` runs all widths 96/192/384 and contexts 128/512/1024, batch 2, 4 variants. Matched held-out Tiny Shakespeare training is included at width 96 and context 128, and a larger accuracy grid is optional.

All results are from **this session only**. Fixed widths do not mean parameter-matched models. Short text experiments do not establish LLM-scale quality. Source code for PPLSTM stays under its AGPL-3.0-only license, and the separate xLSTM code remains under its own upstream license.


## 1 · Imports, GPU detection and settings

In [ ]:
import gc, hashlib, importlib, json, math, os, platform, random, statistics, subprocess, sys, time, traceback, urllib.request, zipfile
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
import pandas as pd
import numpy as np
from IPython.display import display

SEED = 2026
WIDTHS = [96, 192, 384]
CONTEXTS = [128, 512, 1024]
BATCH_SIZE = 2
# For timing stability: 3 independent groups of 5 full training-step measurements
TIMING_ROUNDS = 3
STEPS_PER_ROUND = 5
WARMUP_STEPS = 3
RUN_FULL_SWEEP = True   # 9 width/context pairs × all 4 variants, fresh; graph patch sLSTM only
RUN_ACCURACY = True     # matched text-training experiment at 96×128
RUN_ACCURACY_GRID = False  # optional extra matched-training 3×3 grid; very slow on free T4
ACC_GRID_STEPS = 24        # per model/config, optional grid only
ALLOW_OPTIMIZED_CUDA_BUILD = True  # only attempted if GPU compute capability >= 8.0

# All tests rerun fresh; no historical results are loaded or combined.
# Set this False to avoid the optional GPU accuracy run.
RUN_PARITY = True

# Accuracy experiment: SAME one width/context/steps for all variants.

ACC_WIDTH = 96
ACC_CONTEXT = 128
ACC_BATCH = 8
ACC_STEPS = 80         # pilot only; increase to 400+ for stronger evidence
ACC_VAL_BATCHES = 8
ACC_VAL_BATCH = 8
ACC_LR = 0.001
ACC_SEED = 42
# If desired, change these and rerun the accuracy cell only.

OUTPUT_DIR = Path('/content/pplstm_benchmark_results') if Path('/content').exists() else Path('pplstm_benchmark_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Remove leftover files only in this dedicated results folder; don't mix reruns.
for _old in OUTPUT_DIR.glob('*.csv'): _old.unlink()
for _old in OUTPUT_DIR.glob('*.json'): _old.unlink()
for _old in OUTPUT_DIR.glob('*.zip'): _old.unlink()
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if GPU else 'none'
CAP = torch.cuda.get_device_capability(0) if GPU else (0, 0)
VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if GPU else None
print({'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'device': GPU_NAME,
       'compute_capability': CAP, 'vram_GiB': VRAM_GIB})
if not GPU: print('⚠ Set Runtime > Change runtime type > GPU before speed tests. CPU correctness is still possible.')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if GPU: torch.cuda.manual_seed_all(SEED)
ENV = dict(run_mode='fresh_four_variants_separate_graph_paths',python=sys.version.split()[0],torch=torch.__version__,torch_cuda=torch.version.cuda,
           gpu=GPU_NAME,compute_capability=list(CAP),vram_gib=VRAM_GIB,
           widths=WIDTHS,contexts=CONTEXTS,batch=BATCH_SIZE,
           timing_rounds=TIMING_ROUNDS,timing_steps_per_round=STEPS_PER_ROUND,
           colab=bool(os.environ.get('COLAB_RELEASE_TAG')),
           vanilla_slstm_graph_patch='tensor_where_replaces_host_branch; validation required')


## 2 · Official xLSTM installation

`pip install xlstm` may build/download dependencies. If it fails, PPLSTM experiments still work and **official baseline rows are marked skipped**. We do not silently substitute a custom xLSTM implementation. The official CUDA backend may additionally need a matching CUDA compiler/toolchain; on T4 it is not attempted.

In [ ]:
XLSTM_INSTALLED=False
XLSTM_INSTALL_LOG=''
try:
    import xlstm
    XLSTM_INSTALLED=True
except Exception:
    print('Installing official xlstm package (may require a few minutes)...')
    try:
        p=subprocess.run([sys.executable,'-m','pip','install','-q','xlstm'],capture_output=True,text=True,timeout=480)
        XLSTM_INSTALL_LOG=(p.stdout+'\n'+p.stderr)[-6000:]
        if p.returncode != 0: raise RuntimeError(XLSTM_INSTALL_LOG[-1600:])
        importlib.invalidate_caches()
        import xlstm
        XLSTM_INSTALLED=True
    except Exception as e:
        XLSTM_INSTALL_LOG = (XLSTM_INSTALL_LOG+'\n'+repr(e))[-6000:]
        print('Official xlstm unavailable:',repr(e))
print('official xlstm installed?',XLSTM_INSTALLED)
ENV['xlstm_installed']=XLSTM_INSTALLED
ENV['xlstm_version']=getattr(xlstm,'__version__','unknown') if XLSTM_INSTALLED else None


## 2B · Separate, capture-safe **vanilla sLSTM** path

Your PPLSTM graph remains exactly as before. The vanilla sLSTM **graph-only** model gets a GPU-side replacement for the official host-side condition. We test the new function against the upstream pointwise function on three starting memory states (all-zero, nonzero, mixed) and compare forward values and gradients. Only apply the patch to graph-mode sLSTM instances; eager-mode sLSTM stays unmodified. This is a **benchmark adaptation**, not an upstream xLSTM release or the optimized CUDA sLSTM kernel.


In [ ]:
# Adaptation of a tiny branch in upstream xLSTM vanilla sLSTM pointwise function.
# Upstream file: https://github.com/NX-AI/xlstm/blob/main/xlstm/blocks/slstm/src/vanilla/slstm.py
# Numerical checks below must pass before it is used in any benchmark.

def capture_safe_slstm_pointwise(Wx, Ry, b, states, constants):
    _ = constants
    raw = Wx + Ry + b
    y, c, n, m = torch.unbind(states.view(4, states.shape[1], -1), dim=0)
    iraw, fraw, zraw, oraw = torch.unbind(raw.view(raw.shape[0], 4, -1), dim=1)
    logfplusm = m + F.logsigmoid(fraw)

    # ORIGINAL: if torch.all(n == 0.0): mnew=iraw; else: mnew=torch.max(...)
    # Keep the same all-elements condition, but calculate it entirely on the GPU.
    # This avoids the implicit GPU->CPU boolean synchronization forbidden in capture.
    mnew = torch.where(torch.all(n == 0.0), iraw, torch.max(iraw, logfplusm))

    ogate = torch.sigmoid(oraw)
    igate = torch.minimum(torch.exp(iraw - mnew), torch.ones_like(iraw))
    fgate = torch.minimum(torch.exp(logfplusm - mnew), torch.ones_like(iraw))
    cnew = fgate * c + igate * torch.tanh(zraw)
    nnew = fgate * n + igate
    ynew = ogate * cnew / nnew
    return torch.stack((ynew, cnew, nnew, mnew), dim=0), torch.stack((igate, fgate, zraw, ogate), dim=0)

SLSTM_PATCH_TESTS=[]
SLSTM_PATCH_OK=False

def check_slstm_pointwise_equivalence():
    from xlstm.blocks.slstm.src.vanilla.slstm import slstm_forward_pointwise
    tests=[]
    for case,seed in [('zero_norm',101), ('nonzero_norm',102), ('mixed_norm',103)]:
        rng=torch.Generator(device='cpu').manual_seed(seed)
        shape=(2,8)
        base_wx=torch.randn(2,4*shape[-1],generator=rng)*0.25
        base_ry=torch.randn(2,4*shape[-1],generator=rng)*0.25
        base_b=torch.randn(1,4*shape[-1],generator=rng)*0.25
        base_state=torch.randn(4,*shape,generator=rng)*0.25
        if case=='zero_norm': base_state[2].zero_()
        elif case=='nonzero_norm': base_state[2].fill_(1.0)
        else:
            base_state[2].fill_(1.0); base_state[2,0,0]=0.0
        def eval_fn(fn):
            args=[x.clone().detach().requires_grad_(True) for x in (base_wx,base_ry,base_b,base_state)]
            out=fn(*args,constants={})
            # Weighted sum avoids identical-gradient cancellation across gates.
            score=(out[0]*torch.linspace(0.8,1.3,out[0].numel()).reshape_as(out[0])).sum() + 0.2*out[1].square().sum()
            grads=torch.autograd.grad(score,args)
            return out,grads
        original,grads_orig=eval_fn(slstm_forward_pointwise)
        patched,grads_patch=eval_fn(capture_safe_slstm_pointwise)
        max_out=max(float((a-b).abs().max()) for a,b in zip(original,patched))
        max_grad=max(float((a-b).abs().max()) for a,b in zip(grads_orig,grads_patch))
        ok=all(torch.allclose(a,b,atol=1e-6,rtol=1e-6) for a,b in zip(original,patched)) and \
           all(torch.allclose(a,b,atol=1e-6,rtol=1e-6) for a,b in zip(grads_orig,grads_patch))
        tests.append(dict(test=case,status='PASS' if ok else 'FAIL',max_output_abs=max_out,max_grad_abs=max_grad))
    return tests

if XLSTM_INSTALLED:
    try:
        SLSTM_PATCH_TESTS=check_slstm_pointwise_equivalence()
        SLSTM_PATCH_OK=all(r['status']=='PASS' for r in SLSTM_PATCH_TESTS)
        display(pd.DataFrame(SLSTM_PATCH_TESTS))
        print('sLSTM graph-only patch numerical equivalence:', 'PASS' if SLSTM_PATCH_OK else 'FAIL — sLSTM graph disabled')
    except Exception as e:
        SLSTM_PATCH_TESTS=[dict(test='pointwise_equivalence',status='ERROR',error=f'{type(e).__name__}: {str(e)[:600]}')]
        print('sLSTM graph patch check failed:',SLSTM_PATCH_TESTS[0]['error'])
else:
    print('Cannot check sLSTM patch without the official xlstm package; graph variant will skip.')

def use_capture_safe_pointwise_for_slstm_graph(stack):
    if not SLSTM_PATCH_OK:
        raise RuntimeError('sLSTM graph-only patch blocked: equivalence check did not pass')
    from xlstm.blocks.slstm.cell import sLSTMCell_vanilla
    found=[]
    for module in stack.modules():
        if isinstance(module,sLSTMCell_vanilla):
            module.pointwise=capture_safe_slstm_pointwise
            found.append(module)
    if not found:
        raise RuntimeError('No official sLSTMCell_vanilla found in graph baseline; no capture patch applied')
    return len(found)


## 3 · PPLSTM source code (unchanged mathematical design)

Embedded from your supplied PPLSTM code appendix and previously tested Python transcription, under `AGPL-3.0-only`. We do not replace your three-channel affine scan with an invented kernel. CUDA Graphs change scheduling, not the cell equations.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Copyright (C) 2026 PPLSTM project author
"""PPLSTM: Parallel Permutation Long Short-Term Memory (research prototype).

Transcribed from the user-supplied PPLSTM code appendix (November 2026).
Input: [batch, time, width], with width divisible by three.
This is not mathematically equivalent to an sLSTM.
"""
from __future__ import annotations
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    """Inclusive scan for x <- a*x + b, with scalar coefficients."""
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    """Inclusive scan of 3-channel affine maps, composed left-to-right."""
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    """Permutation-mixed signed-memory cell with parallel affine scans."""
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError("width must be a positive multiple of 3")
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer("permutation_bank", torch.stack(bank))
        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError("Expected x with shape [batch, time, width]")
        batch, length, width = x.shape
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (
            self.xproj(x).reshape(batch, length, 4, width) * coeff
            + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff)
        )
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, self.groups, 3)
        z = candidate_logits.tanh().reshape(batch, length, self.groups, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, self.groups, 3).mean(-1)
        log_f = F.logsigmoid(
            (forget_logits + self.forget_bias).reshape(batch, length, self.groups, 3).mean(-1)
        )
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(
            torch.cat((torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix), dim=1), dim=1
        ).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, self.groups, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio


class PPLSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size: int = 512, width: int = 36):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, width)
        self.norm = nn.LayerNorm(width)
        self.cell = PPLSTM(width)
        self.proj = nn.Linear(width, width)
        self.output_norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def forward(self, token_ids: Tensor) -> Tensor:
        x = self.embedding(token_ids) / math.sqrt(self.embedding.embedding_dim)
        x = x + self.proj(self.cell(self.norm(x)))
        return self.head(self.output_norm(x))


## 4 · Shared backbone and task wrapper

In [ ]:
class PPLSTMLayerAdapter(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.cell=PPLSTM(width)
        self.output_norm=nn.LayerNorm(width)
    def forward(self,x,**kwargs):
        if kwargs.get('return_last_state',False):
            raise NotImplementedError('This research PPLSTM has no streaming state API')
        return self.output_norm(self.cell(x))
    def reset_parameters(self): pass

class FallbackResidualPPLSTM(nn.Module):
    # Used ONLY if official xlstm package is missing; labelled in results.
    def __init__(self,width):
        super().__init__()
        self.norm1=nn.LayerNorm(width)
        self.inner=PPLSTMLayerAdapter(width)
        self.norm2=nn.LayerNorm(width)
        self.ff=nn.Sequential(nn.Linear(width,2*width),nn.GELU(),nn.Linear(2*width,width))
    def forward(self,x):
        x=x+self.inner(self.norm1(x))
        return x+self.ff(self.norm2(x))

def build_backbone(variant,width,context):
    if width % 12: raise ValueError('Width must be a multiple of 12')
    if variant=='xlstm_cuda' and CAP<(8,0):
        raise RuntimeError('Optimized official sLSTM CUDA requires compute capability >= 8.0')
    if not XLSTM_INSTALLED:
        if variant.startswith('pplstm_'):
            return FallbackResidualPPLSTM(width),'STANDALONE PPLSTM wrapper (official xlstm unavailable)'
        raise RuntimeError('Official xlstm installation unavailable')
    from xlstm import xLSTMBlockStack,xLSTMBlockStackConfig,sLSTMBlockConfig,sLSTMLayerConfig,FeedForwardConfig
    backend='cuda' if variant=='xlstm_cuda' else 'vanilla'
    cfg=xLSTMBlockStackConfig(
        mlstm_block=None,
        slstm_block=sLSTMBlockConfig(
            slstm=sLSTMLayerConfig(backend=backend,num_heads=4,
                                   conv1d_kernel_size=0,bias_init='powerlaw_blockdependent'),
            feedforward=FeedForwardConfig(proj_factor=1.3,act_fn='gelu')),
        embedding_dim=width,context_length=context,num_blocks=1,slstm_at=[0],dropout=0.0)
    stack=xLSTMBlockStack(cfg)
    if variant=='xlstm_vanilla_graph':
        use_capture_safe_pointwise_for_slstm_graph(stack)
    if variant.startswith('pplstm_'):
        stack.blocks[0].xlstm=PPLSTMLayerAdapter(width)
    return stack, ('official xLSTM block; vanilla sLSTM capture-safe tensor branch, sLSTM conv disabled'
                   if variant=='xlstm_vanilla_graph' else
                   'official xLSTM one-block residual wrapper; sLSTM conv disabled')

class CharLM(nn.Module):
    def __init__(self, core, width, vocab):
        super().__init__()
        self.embedding=nn.Embedding(vocab,width)
        self.core=core
        self.output_norm=nn.LayerNorm(width)
        self.head=nn.Linear(width,vocab,bias=False)
        self.head.weight=self.embedding.weight
        self.width=width
    def forward(self,tokens):
        x=self.embedding(tokens)/math.sqrt(self.width)
        y=self.core(x)
        return self.head(self.output_norm(y))/math.sqrt(self.width)

def build_char_model(variant,width,context,vocab,seed):
    torch.manual_seed(seed)
    core,wrapper=build_backbone(variant,width,context)
    m=CharLM(core,width,vocab).to(DEVICE).float()
    return m,wrapper

def nparams(model): return sum(p.numel() for p in model.parameters())

def cleanup():
    gc.collect()
    if GPU: torch.cuda.empty_cache()


## 5 · Numerical correctness / gradient stability

Compare your *parallel* recurrence against the **serial reference of your own equations**; this does **not** imply mathematical equality to sLSTM. Check outputs, input and parameter gradients, relative error and finiteness at short and longer lengths. Failure rows remain in the export. 1e-4 output and 5e-4 gradient absolute tolerances are diagnostic thresholds (FP32 arithmetic order can differ).

In [ ]:
CORRECTNESS_ROWS=[]
def scan_gradient_check(width,length,batch=2,seed=13):
    row=dict(test='parallel_vs_serial',width=width,context=length,device=DEVICE,
             status='FAILED',output_max_abs=None,gradient_max_abs=None,gradient_max_scaled=None,
             all_finite=False,error='')
    try:
        torch.manual_seed(seed)
        cell=PPLSTM(width).to(DEVICE).float()
        x=torch.randn(batch,length,width,device=DEVICE,requires_grad=True)
        parallel=cell(x); serial=cell.serial_reference(x)
        args=tuple(cell.parameters())+(x,)
        gp=torch.autograd.grad(parallel.square().mean(),args,retain_graph=True)
        gs=torch.autograd.grad(serial.square().mean(),args)
        row['output_max_abs']=float((parallel-serial).abs().max().detach())
        row['gradient_max_abs']=max(float((a-b).abs().max().detach()) for a,b in zip(gp,gs))
        row['gradient_max_scaled']=max(float(((a-b).abs()/torch.maximum(torch.maximum(a.abs(),b.abs()),torch.tensor(1e-3,device=DEVICE))).max().detach()) for a,b in zip(gp,gs))
        row['all_finite']=bool(torch.isfinite(parallel).all().item() and torch.isfinite(serial).all().item() and all(torch.isfinite(z).all().item() for z in gp+gs))
        row['status']='PASS' if row['all_finite'] and row['output_max_abs']<1e-4 and row['gradient_max_abs']<5e-4 else 'CHECK'
    except Exception as e: row['error']=f'{type(e).__name__}: {str(e)[:700]}'
    return row

for width,context in [(12,17),(96,128),(96,512),(96,1024)]:
    print('Checking PPLSTM recurrence',width,context,flush=True)
    CORRECTNESS_ROWS.append(scan_gradient_check(width,context))
display(pd.DataFrame(CORRECTNESS_ROWS))


## 6 · CUDA Graph training-step infrastructure

Both PPLSTM and **official vanilla xLSTM** are run with and without CUDA Graph replay. This preserves the memory cell and architecture while changing how the operations are dispatched. Both modes use the same width, objective, dtype, optimizer (`AdamW`, `foreach=False`), input shapes and random seed. Graph capture includes **forward + backward + optimizer step**. Capture/warmup costs are separate from steady-state replay timing.

For accuracy, graph capture/warmup **temporarily updates weights**, then **restores every parameter** and zeroes Adam moments/step counters *in-place* before the first counted training batch. This avoids giving graph training an unfair warmup advantage. Static inputs/targets are copied in before each replay. Graph failures are clearly reported, never silently replaced by eager mode. Graph performance measures host-observed full-step latency and its Python launch overhead, not a new fused CUDA kernel.

In [ ]:
class TrainStepper:
    def __init__(self,model,variant,lr,first_x,first_y=None,mode='text'):
        if mode not in ('text','regression'): raise ValueError(mode)
        self.model=model
        self.variant=variant
        self.mode=mode
        self.use_graph=(variant in ('pplstm_graph', 'xlstm_vanilla_graph'))
        self.optim=torch.optim.AdamW(model.parameters(),lr=lr,foreach=False,
                                     capturable=bool(GPU))
        self.graph=None; self.graph_loss=None
        self.fixed_x=first_x.clone()
        self.fixed_y=first_y.clone() if first_y is not None else None
        self.setup_s=0.0
        begin=time.perf_counter()
        if self.use_graph:
            if not GPU: raise RuntimeError('CUDA Graphs require an NVIDIA CUDA GPU')
            # Snapshot exact initial model params; restore after warmup + capture.
            initial=[p.detach().clone() for p in model.parameters()]
            self._step()
            torch.cuda.synchronize()
            side=torch.cuda.Stream()
            side.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(side):
                for _ in range(3): self._step()
            torch.cuda.current_stream().wait_stream(side)
            torch.cuda.synchronize()
            self.graph=torch.cuda.CUDAGraph()
            with torch.cuda.graph(self.graph):
                self.graph_loss=self._step()
            # Crucial: DON'T replace the tensors. The captured graph keeps their addresses.
            with torch.no_grad():
                for p,original in zip(model.parameters(),initial): p.copy_(original)
                for state in self.optim.state.values():
                    for v in state.values():
                        if isinstance(v,torch.Tensor): v.zero_()
            torch.cuda.synchronize()
            del initial
        self.setup_s=time.perf_counter()-begin

    def _step(self):
        self.optim.zero_grad(set_to_none=False)
        output=self.model(self.fixed_x)
        if self.mode=='text':
            loss=F.cross_entropy(output.reshape(-1,output.shape[-1]),self.fixed_y.reshape(-1))
        else:
            loss=output.square().mean()
        loss.backward()
        self.optim.step()
        return loss

    def step(self,x,y=None):
        if self.use_graph:
            self.fixed_x.copy_(x)
            if y is not None: self.fixed_y.copy_(y)
            self.graph.replay()
            return self.graph_loss
        self.fixed_x=x
        self.fixed_y=y
        return self._step()

def build_stepper(variant,width,context,batch,mode='regression',lr=0.001,seed=SEED,vocab=None,first_x=None,first_y=None):
    torch.manual_seed(seed)
    if mode=='regression':
        m,wrapper=build_backbone(variant,width,context)
        m=m.to(DEVICE).float().train()
        if first_x is None: first_x=torch.randn(batch,context,width,device=DEVICE)
    else:
        if vocab is None: raise ValueError('vocab required for text')
        m,wrapper=build_char_model(variant,width,context,vocab,seed)
        m.train()
        if first_x is None or first_y is None: raise ValueError('text needs initial x and y')
    stepper=TrainStepper(m,variant,lr,first_x,first_y,mode)
    return m,stepper,wrapper


## 7 · Graph-vs-eager **one-step numerical agreement**, both architectures

Construct identical PPLSTM **and official vanilla xLSTM** language models from the same seed, use the **same minibatch**, execute exactly one counted optimizer step, and compare pre/post-weights plus gradients. Minor numerical differences may occur from CUDA scheduling. This is separate from the serial-scan check above. If graph capture fails, capture the error; the other tests still run.

In [ ]:
GRAPH_PARITY=[]
def graph_eager_agreement(base='pplstm',width=96,context=128,batch=2,vocab=67):
    if base=='pplstm': eager_variant,graph_variant='pplstm_eager','pplstm_graph'
    elif base=='xlstm_vanilla': eager_variant,graph_variant='xlstm_vanilla','xlstm_vanilla_graph'
    else: raise ValueError(base)
    row=dict(test='graph_vs_eager_step',architecture=base,graph_variant=graph_variant,
             width=width,context=context,status='FAILED',
             initial_weights_max_abs=None,loss_max_abs=None,grad_max_abs=None,
             weights_after_step_max_abs=None,error='')
    if not GPU: row.update(status='SKIPPED',error='No CUDA GPU'); return row
    if base=='xlstm_vanilla' and not XLSTM_INSTALLED:
        row.update(status='SKIPPED',error='Official xlstm not installed'); return row
    eager_m=graph_m=eager=graph=None
    try:
        rng=torch.Generator().manual_seed(977)
        tokens=torch.randint(0,vocab,(batch,context),generator=rng).to(DEVICE)
        targets=torch.randint(0,vocab,(batch,context),generator=rng).to(DEVICE)
        eager_m,eager,_=build_stepper(eager_variant,width,context,batch,mode='text',seed=51,vocab=vocab,first_x=tokens,first_y=targets)
        graph_m,graph,_=build_stepper(graph_variant,width,context,batch,mode='text',seed=51,vocab=vocab,first_x=tokens,first_y=targets)
        a=list(eager_m.parameters()); b=list(graph_m.parameters())
        row['initial_weights_max_abs']=max(float((p-q).abs().max()) for p,q in zip(a,b))
        la=eager.step(tokens,targets); lb=graph.step(tokens,targets)
        torch.cuda.synchronize()
        row['loss_max_abs']=abs(float(la.detach())-float(lb.detach()))
        row['grad_max_abs']=max(float((p.grad-q.grad).abs().max()) for p,q in zip(a,b) if p.grad is not None and q.grad is not None)
        row['weights_after_step_max_abs']=max(float((p-q).abs().max()) for p,q in zip(a,b))
        row['status']='PASS' if row['initial_weights_max_abs'] < 1e-7 and row['loss_max_abs'] < 1e-3 and row['grad_max_abs'] < 1e-3 and row['weights_after_step_max_abs'] < 1e-3 else 'CHECK'
    except Exception as e:
        row['error']=f'{type(e).__name__}: {str(e)[:800]}'
        print('PARITY',base,'FAILED:',row['error'])
    finally:
        del eager_m,graph_m,eager,graph
        cleanup()
    return row

if RUN_PARITY:
    # Check one-step graph/eager parity for BOTH architectures.
    for base in ['pplstm','xlstm_vanilla']:
        print('Graph parity:',base,flush=True)
        GRAPH_PARITY.append(graph_eager_agreement(base=base))
display(pd.DataFrame(GRAPH_PARITY))


## 8 · Repeated **full training-step speed** and memory

Each run times **forward + backward + AdamW + GPU synchronization** on a fixed tensor; 3 rounds × 5 timed steps, with untimed warmups. We record the median, mean, p10–p90 range, standard deviation, coefficient of variation and round medians. Tokens/sec = batch × context / step wall time. CUDA Graph capture/setup is reported separately. Memory is `torch.cuda.max_memory_allocated`, **not** board-wide VRAM use. Shape-matched comparisons also report parameter counts.

On a T4, `xlstm_cuda` is **SKIPPED**. A failed attempt to build the official CUDA kernel is **FAILED**, not a fallback to `vanilla`. The optional official-optimized baseline is attempted once on a compatible GPU, then included in the sweep only if it passes.

In [ ]:
SPEED_ROWS=[]  # this run ONLY
OPTIMIZED_BASELINE_STATUS='not attempted'
def speed_one(variant,width,context,batch=BATCH_SIZE,rounds=TIMING_ROUNDS,reps=STEPS_PER_ROUND):
    row=dict(variant=variant,width=width,context=context,batch=batch,status='FAILED',
             params=None,wrapper='',source_run='current_colab',median_ms=None,mean_ms=None,std_ms=None,cv_pct=None,
             p10_ms=None,p90_ms=None,round_medians_ms=None,tokens_per_s=None,
             peak_alloc_mib=None,setup_s=None,last_loss_finite=None,error='')
    if not GPU:
        row.update(status='SKIPPED',error='CUDA GPU unavailable'); return row
    if variant=='xlstm_cuda' and CAP<(8,0):
        row.update(status='SKIPPED',error='Official optimized sLSTM CUDA requires compute capability >= 8.0'); return row
    if variant=='xlstm_vanilla_graph' and not SLSTM_PATCH_OK:
        row.update(status='SKIPPED',error='capture-safe sLSTM pointwise numerical check did not pass'); return row
    if variant.startswith('xlstm_') and not XLSTM_INSTALLED:
        row.update(status='SKIPPED',error='Official xlstm not installed'); return row
    model=runner=None
    try:
        model,runner,wrapper=build_stepper(variant,width,context,batch)
        row['wrapper']=wrapper
        row['params']=nparams(model)
        row['setup_s']=round(runner.setup_s,4)
        # Graph replay and eager step both update their real parameters.
        for _ in range(WARMUP_STEPS): runner.step(runner.fixed_x)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        times=[]; round_medians=[]; loss=None
        for r in range(rounds):
            vals=[]
            for i in range(reps):
                torch.cuda.synchronize()
                tic=time.perf_counter()
                loss=runner.step(runner.fixed_x)
                torch.cuda.synchronize()
                vals.append(1000*(time.perf_counter()-tic))
            round_medians.append(round(statistics.median(vals),4))
            times.extend(vals)
        med=statistics.median(times)
        row.update(status='PASS',median_ms=round(med,4),mean_ms=round(statistics.mean(times),4),
                   std_ms=round(statistics.stdev(times) if len(times)>1 else 0.,4),
                   cv_pct=round(100*statistics.stdev(times)/statistics.mean(times),2) if len(times)>1 else 0.,
                   p10_ms=round(float(np.percentile(times,10)),4),p90_ms=round(float(np.percentile(times,90)),4),
                   round_medians_ms=round_medians,tokens_per_s=round(batch*context*1000/med,1),
                   peak_alloc_mib=round(torch.cuda.max_memory_allocated()/2**20,2),
                   last_loss_finite=bool(torch.isfinite(loss.detach()).all().item()))
        if not row['last_loss_finite']: row['status']='NONFINITE'
    except Exception as e:
        row['error']=f'{type(e).__name__}: {str(e)[:1000]}'
        if 'out of memory' in row['error'].lower(): row['status']='OOM'
        print('BENCHMARK',variant,width,context,row['status'],row['error'],flush=True)
    finally:
        del runner,model
        if GPU:
            try: torch.cuda.synchronize()
            except Exception: pass
        cleanup()
    return row

def run_benchmark_grid(configs,variants):
    existing={(r['variant'],r['width'],r['context']) for r in SPEED_ROWS}
    for width,context in configs:
        for variant in variants:
            if (variant,width,context) in existing:
                print('Already measured this session:',variant,width,context)
                continue
            print('Speed:',variant,'width',width,'context',context,flush=True)
            row=speed_one(variant,width,context)
            SPEED_ROWS.append(row)
            existing.add((variant,width,context))
            display(pd.DataFrame([row])[['variant','width','context','status','median_ms','cv_pct','tokens_per_s','peak_alloc_mib','error']])
            save_outputs() if 'save_outputs' in globals() else None


### 8A · Four-way smoke test (fresh run)

Start with all four variants at **96 channels × 128 tokens**. If a CUDA Graph capture fails, the row is labelled `FAILED` with its error message.

In [ ]:
SMOKE_CONFIG=(96,128)
if GPU:
    run_benchmark_grid([SMOKE_CONFIG],['pplstm_eager','pplstm_graph','xlstm_vanilla','xlstm_vanilla_graph'])
else:
    print('GPU unavailable; smoke speed test skipped')


### 8B · Probe **official optimized CUDA** only when GPU allows it

In [ ]:
CAN_RUN_OPTIMIZED = (GPU and XLSTM_INSTALLED and CAP >= (8,0) and ALLOW_OPTIMIZED_CUDA_BUILD)
if CAN_RUN_OPTIMIZED:
    print('Testing official xLSTM CUDA build at 96×128; compilation may take time.')
    optimized_probe=speed_one('xlstm_cuda',96,128,rounds=1,reps=3)
    SPEED_ROWS.append(optimized_probe)
    OPTIMIZED_BASELINE_STATUS=optimized_probe['status']
    display(pd.DataFrame([optimized_probe]))
else:
    why='GPU compute capability below 8.0' if CAP < (8,0) else 'xlstm missing or CUDA build disabled'
    OPTIMIZED_BASELINE_STATUS='SKIPPED: '+why
    SPEED_ROWS.append(dict(variant='xlstm_cuda',width=96,context=128,batch=BATCH_SIZE,
                           status='SKIPPED',error=why))
    print('Optimized official xLSTM:',OPTIMIZED_BASELINE_STATUS)


### 8C · Full 9-configuration × 4-variant fresh speed sweep

Widths `[96, 192, 384]`, contexts `[128, 512, 1024]`, batch 2, repeated full training steps (forward + backward + AdamW). The 96×128 smoke results are re-used **only within this same run** to avoid duplicate measurements. Longer vanilla xLSTM runs may take several minutes. Missing optimized kernel support is recorded as `SKIPPED`. CUDA Graph capture failures are recorded as `FAILED`.

In [ ]:
if RUN_FULL_SWEEP and GPU:
    variants=['pplstm_eager','pplstm_graph','xlstm_vanilla','xlstm_vanilla_graph']
    if OPTIMIZED_BASELINE_STATUS=='PASS':
        variants.append('xlstm_cuda') # opportunistic official optimized baseline on compatible GPU
    run_benchmark_grid([(w,c) for w in WIDTHS for c in CONTEXTS],variants)
else:
    print('Full sweep not run (RUN_FULL_SWEEP=False or no GPU).')


## 9 · Matched *held-out text* training and validation

This is **real character-level next-character prediction**, not the synthetic speed objective. We use the [Tiny Shakespeare file](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare) with a deterministic contiguous **90% train / 10% validation split**; vocabulary is built from training text only, with an explicit unknown-character token for validation. All variants use **identical batches**, **same optimization steps (default 80)**, **same seed, LR, AdamW setup, validation offsets**. Save both validation loss and character accuracy (plus perplexity); evaluate at step 0 and at the end. **PPLSTM eager and graph must start from identical parameters**, as verified above. Graph capture/warmup never counts as training.

This is a **pilot**: only one width/context and one seed unless you change settings. More steps/seeds and parameter-matched comparisons are essential before making strong quality claims. If Tiny Shakespeare cannot be downloaded, report SKIPPED; no fake validation result.

In [ ]:
TEXT_DATA={}; ACCURACY_ROWS=[]; ACCURACY_CURVES=[] # freshly measured only
def prepare_tiny_shakespeare():
    url='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with urllib.request.urlopen(url,timeout=25) as f:
        raw=f.read(3_000_000)
    decoded=raw.decode('utf-8')
    boundary=int(len(decoded)*0.9)
    tr=decoded[:boundary]; va=decoded[boundary:]
    alphabet=sorted(set(tr))
    toidx={ch:i+1 for i,ch in enumerate(alphabet)}  # 0 is UNK
    train=torch.tensor([toidx.get(ch,0) for ch in tr],dtype=torch.long)
    valid=torch.tensor([toidx.get(ch,0) for ch in va],dtype=torch.long)
    info=dict(url=url,sha256=hashlib.sha256(raw).hexdigest(),n_train=len(train),n_valid=len(valid),
              vocab=len(alphabet)+1,unknown_validation_chars=int((valid==0).sum().item()))
    return train,valid,info

if RUN_ACCURACY:
    try:
        train_ids,val_ids,dataset_info=prepare_tiny_shakespeare()
        TEXT_DATA={'train':train_ids,'valid':val_ids,'info':dataset_info}
        print('Tiny Shakespeare:',dataset_info)
    except Exception as e:
        TEXT_DATA={'error':f'{type(e).__name__}: {e}'}
        print('Accuracy data unavailable:',TEXT_DATA['error'])
else:
    print('Accuracy disabled by RUN_ACCURACY=False')


In [ ]:
def fixed_text_batches(ids,steps,batch,context,seed,device=DEVICE):
    # On CPU generator: independent of GPU state and identical for every variant.
    gen=torch.Generator(device='cpu').manual_seed(seed)
    top=len(ids)-context-1
    if top <= 0: raise ValueError('Text split shorter than context')
    starts=torch.randint(0,top,(steps,batch),generator=gen)
    offsets=torch.arange(context)
    return [(ids[(s[:,None]+offsets)].to(device),
             ids[(s[:,None]+offsets+1)].to(device)) for s in starts]

# Prepare ONE fixed order of train batches and ONE separate fixed held-out batch set.
if 'train' in TEXT_DATA:
    ACC_TRAIN_BATCHES=fixed_text_batches(TEXT_DATA['train'],ACC_STEPS,ACC_BATCH,ACC_CONTEXT,ACC_SEED+10)
    ACC_VALID_BATCHES=fixed_text_batches(TEXT_DATA['valid'],ACC_VAL_BATCHES,ACC_VAL_BATCH,ACC_CONTEXT,ACC_SEED+200)
else:
    ACC_TRAIN_BATCHES=[]; ACC_VALID_BATCHES=[]

@torch.no_grad()
def evaluate_char_model(model,val_batches):
    model.eval()
    nll=0.0; hits=0; count=0
    for x,y in val_batches:
        logits=model(x)
        nll += float(F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1),reduction='sum'))
        hits += int((logits.argmax(-1)==y).sum().item())
        count += y.numel()
    model.train()
    avg=nll/count
    return dict(val_nll=avg,val_ppl=math.exp(min(50.,avg)),val_accuracy=hits/count,eval_positions=count)

def train_char_variant(variant):
    row=dict(variant=variant,status='FAILED',width=ACC_WIDTH,context=ACC_CONTEXT,
             steps=ACC_STEPS,batch=ACC_BATCH,params=None,source_run='current_colab',initial_val_nll=None,
             initial_val_accuracy=None,final_val_nll=None,final_val_ppl=None,
             final_val_accuracy=None,train_last_nll=None,train_wall_s=None,
             train_tokens_per_s=None,grad_norm_first=None,grad_all_finite=None,
             wrapper='',error='')
    if not ACC_TRAIN_BATCHES: row.update(status='SKIPPED',error='No text dataset'); return row
    if variant=='xlstm_cuda' and CAP<(8,0): row.update(status='SKIPPED',error='CUDA capability < 8.0'); return row
    if variant=='xlstm_vanilla_graph' and not SLSTM_PATCH_OK:
        row.update(status='SKIPPED',error='capture-safe sLSTM pointwise numerical check did not pass'); return row
    if variant.startswith('xlstm_') and not XLSTM_INSTALLED: row.update(status='SKIPPED',error='official xlstm missing'); return row
    model=runner=None
    try:
        first_x,first_y=ACC_TRAIN_BATCHES[0]
        model,runner,wrapper=build_stepper(variant,ACC_WIDTH,ACC_CONTEXT,ACC_BATCH,
            mode='text',lr=ACC_LR,seed=ACC_SEED,vocab=TEXT_DATA['info']['vocab'],
            first_x=first_x,first_y=first_y)
        row['wrapper']=wrapper; row['params']=nparams(model)
        init=evaluate_char_model(model,ACC_VALID_BATCHES)
        row['initial_val_nll']=round(init['val_nll'],6)
        row['initial_val_accuracy']=round(init['val_accuracy'],6)
        started=time.perf_counter()
        grad_finite=True; first_grad_norm=None
        for i,(x,y) in enumerate(ACC_TRAIN_BATCHES):
            loss=runner.step(x,y)
            if i==0:
                # synchronize only for diagnostic extraction; no gradient clipping used.
                grads=[p.grad.detach() for p in model.parameters() if p.grad is not None]
                grad_finite=all(bool(torch.isfinite(g).all().item()) for g in grads)
                first_grad_norm=math.sqrt(sum(float(g.float().square().sum()) for g in grads))
            if i in (0,ACC_STEPS//2,ACC_STEPS-1):
                if GPU: torch.cuda.synchronize()
                ACCURACY_CURVES.append(dict(variant=variant,source_run='current_colab',step=i+1,train_nll=round(float(loss.detach()),6)))
        if GPU: torch.cuda.synchronize()
        elapsed=time.perf_counter()-started
        final=evaluate_char_model(model,ACC_VALID_BATCHES)
        row.update(status='PASS',final_val_nll=round(final['val_nll'],6),
                   final_val_ppl=round(final['val_ppl'],6),
                   final_val_accuracy=round(final['val_accuracy'],6),
                   train_last_nll=round(float(loss.detach()),6),
                   train_wall_s=round(elapsed,3),
                   train_tokens_per_s=round(ACC_BATCH*ACC_CONTEXT*ACC_STEPS/elapsed,1),
                   grad_norm_first=round(first_grad_norm,6),grad_all_finite=grad_finite)
        if not math.isfinite(row['final_val_nll']) or not grad_finite: row['status']='NONFINITE'
    except Exception as e:
        row['error']=f'{type(e).__name__}: {str(e)[:1000]}'
        print('ACCURACY ERROR:',variant,row['error'])
    finally:
        del model,runner
        cleanup()
    return row

if RUN_ACCURACY and ACC_TRAIN_BATCHES:
    acc_variants=['pplstm_eager','pplstm_graph','xlstm_vanilla','xlstm_vanilla_graph']
    if OPTIMIZED_BASELINE_STATUS=='PASS': acc_variants.append('xlstm_cuda')
    for v in acc_variants:
        print('Train+validate:',v,'on identical Tiny Shakespeare batches',flush=True)
        ACCURACY_ROWS.append(train_char_variant(v))
        display(pd.DataFrame([ACCURACY_ROWS[-1]])[['variant','status','steps','initial_val_accuracy',
                      'final_val_accuracy','final_val_nll','final_val_ppl','train_wall_s','error']])
else:
    print('Accuracy training skipped: RUN_ACCURACY=False or dataset download failed')


### 9B · Optional **matched-learning 3 × 3 grid** (resource-intensive)

The default held-out text study above trains all feasible variants at **96 × 128** for 80 steps. To additionally train/evaluate **each of the nine width/context pairs**, set `RUN_ACCURACY_GRID=True` at the top. The optional grid uses `ACC_GRID_STEPS=24` per model/config by default to reduce free GPU usage; this is a **smoke-level quality test**, not enough for convincing learning conclusions. It uses the same fixed training batches, held-out samples, optimizer, seed, and steps **within each configuration**. Validation rows include their width/context/step budget, so you won't mix 24-step results with 80-step results. On a T4, official xLSTM vanilla at long sequence lengths can make this very slow.

In [ ]:
if RUN_ACCURACY_GRID and 'train' in TEXT_DATA:
    saved=(ACC_WIDTH,ACC_CONTEXT,ACC_BATCH,ACC_STEPS,ACC_VAL_BATCHES,ACC_VAL_BATCH,
           ACC_TRAIN_BATCHES,ACC_VALID_BATCHES)
    try:
        for grid_width in WIDTHS:
            for grid_context in CONTEXTS:
                ACC_WIDTH=grid_width
                ACC_CONTEXT=grid_context
                ACC_BATCH=2
                ACC_STEPS=ACC_GRID_STEPS
                ACC_VAL_BATCHES=2
                ACC_VAL_BATCH=2
                ACC_TRAIN_BATCHES=fixed_text_batches(TEXT_DATA['train'],ACC_STEPS,ACC_BATCH,ACC_CONTEXT,ACC_SEED+10)
                ACC_VALID_BATCHES=fixed_text_batches(TEXT_DATA['valid'],ACC_VAL_BATCHES,ACC_VAL_BATCH,ACC_CONTEXT,ACC_SEED+200)
                for variant in (['pplstm_eager','pplstm_graph','xlstm_vanilla','xlstm_vanilla_graph'] +
                                (['xlstm_cuda'] if OPTIMIZED_BASELINE_STATUS=='PASS' else [])):
                    print('Accuracy GRID:',variant,grid_width,grid_context,flush=True)
                    result=train_char_variant(variant)
                    result['protocol']='optional_grid'
                    ACCURACY_ROWS.append(result)
                    display(pd.DataFrame([result])[['variant','width','context','steps','status','final_val_nll','final_val_accuracy','error']])
    finally:
        (ACC_WIDTH,ACC_CONTEXT,ACC_BATCH,ACC_STEPS,ACC_VAL_BATCHES,ACC_VAL_BATCH,
         ACC_TRAIN_BATCHES,ACC_VALID_BATCHES)=saved
else:
    print('Optional 3×3 matched-learning grid not run. Enable RUN_ACCURACY_GRID=True for all 9 configurations.')


## 10 · Same-session results and exports

All benchmark rows from this notebook were measured in the **current Colab run**; no old results are used. Speedup ratios compare rows with matching widths and contexts.  

`xlstm_vanilla_graph` is **CUDA Graph replay of vanilla xLSTM**; it does **not** enable the fused optimized xLSTM sLSTM CUDA kernel. Widths match but parameter counts do not; one residual block is tested. The Tiny Shakespeare learning comparison is short and uses one seed.

Each CSV is included in a final JSON and ZIP export, with GPU/software metadata. If the GPU notebook stops early, the final export cell can still save measurements collected up to that point.

In [ ]:
def save_outputs():
    paths={}
    collections={
        'speed':SPEED_ROWS,'scan_correctness':CORRECTNESS_ROWS,
        'graph_parity':GRAPH_PARITY,'accuracy':ACCURACY_ROWS,
        'accuracy_curve':ACCURACY_CURVES,
        'slstm_graph_patch_equivalence':SLSTM_PATCH_TESTS}
    for name,rows in collections.items():
        path=OUTPUT_DIR/f'{name}.csv'; pd.DataFrame(rows).to_csv(path,index=False); paths[name]=str(path)
    meta=dict(environment=ENV,
           mode='fresh_four_variants_separate_graph_paths',
           current_gpu=GPU_NAME,
           cross_session_comparisons_uncontrolled=False,
           settings=dict(widths=WIDTHS,contexts=CONTEXTS,batch=BATCH_SIZE,
           timing_rounds=TIMING_ROUNDS,timing_steps=STEPS_PER_ROUND,
           accuracy_width=ACC_WIDTH,accuracy_context=ACC_CONTEXT,accuracy_batch=ACC_BATCH,
           accuracy_steps=ACC_STEPS,accuracy_seed=ACC_SEED,accuracy_lr=ACC_LR),
           dataset=TEXT_DATA.get('info',TEXT_DATA.get('error','not run')),
           optimized_xlstm_status=OPTIMIZED_BASELINE_STATUS,
           notes=['PPLSTM replaces one sLSTM position in an official xLSTM block; not a full mLSTM+sLSTM model.',
                  'Width-matched, NOT parameter-matched.',
                  'xlstm_vanilla_graph uses an equivalence-checked host-branch-to-torch.where patch ONLY in the vanilla sLSTM graph cell, NOT the optimized xLSTM CUDA kernel.',
                  'xlstm_cuda only corresponds to optimized sLSTM if compilation and runtime succeeded.',
                  'CUDA Graph speedups reduce launch overhead; they are not fused recurrence optimizations.',
                  'All results measured in this notebook session (no previous results imported).',
                  'Accuracy is a single-seed short Tiny Shakespeare character-level pilot.'],
           results=collections)
    out=OUTPUT_DIR/'pplstm_xlstm_separate_cuda_graphs_results.json'
    out.write_text(json.dumps(meta,indent=2,default=str)); paths['json']=str(out)
    zip_path=OUTPUT_DIR/'PPLSTM_xLSTM_Separate_CUDA_Graphs_Results.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for f in OUTPUT_DIR.glob('*.csv'): z.write(f,arcname=f.name)
        z.write(out,arcname=out.name)
    paths['zip']=str(zip_path)
    return paths

if SPEED_ROWS:
    table=pd.DataFrame(SPEED_ROWS)
    cols=['variant','width','context','status','source_run','params','median_ms','cv_pct',
          'tokens_per_s','peak_alloc_mib','error']
    display(table[[c for c in cols if c in table.columns]])
    good=table[(table['status']=='PASS') & table['median_ms'].notna()]
    if not good.empty:
        pivot=good.pivot_table(index=['width','context'],columns='variant',values='median_ms',aggfunc='first')
        if 'xlstm_vanilla_graph' in pivot and 'xlstm_vanilla' in pivot:
            pivot['vanilla_graph_speedup_over_vanilla_eager']=(pivot['xlstm_vanilla']/pivot['xlstm_vanilla_graph'])
        if 'xlstm_vanilla_graph' in pivot and 'pplstm_graph' in pivot:
            pivot['xlstm_graph_time_over_pplstm_graph']=(pivot['xlstm_vanilla_graph']/pivot['pplstm_graph'])
        if 'pplstm_graph' in pivot and 'pplstm_eager' in pivot:
            pivot['pplstm_graph_speedup_over_eager']=(pivot['pplstm_eager']/pivot['pplstm_graph'])
        display(pivot.round(3))
if ACCURACY_ROWS:
    df=pd.DataFrame(ACCURACY_ROWS)
    display(df[[c for c in ['variant','status','source_run','params','final_val_nll',
                            'final_val_ppl','final_val_accuracy','train_tokens_per_s','error'] if c in df.columns]])
PATHS=save_outputs()
print('Saved:')
for name,path in PATHS.items(): print(name,':',path)
DOWNLOAD_ZIP=False  # flip True to trigger browser download automatically
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(PATHS['zip'])
    except Exception as e: print('Download prompt unavailable:',e)


### Download your full fresh-run results

Colab → Files → `pplstm_benchmark_results` → **`PPLSTM_xLSTM_Separate_CUDA_Graphs_Results.zip`**. The archive includes a CSV for the **vanilla sLSTM pointwise equivalence test**, numerical parity, all four speed variants, memory, and short Tiny Shakespeare validation. Upload the ZIP here to update your PDF with properly supported graph-vs-graph speedups.

The patched graph run still uses the official **vanilla** sLSTM recurrent loop; it does not use the official optimized fused CUDA sLSTM kernel. If the graph baseline still fails, the raw failure is recorded instead of inventing a speedup.
